In [64]:
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
import yfinance as yf
import scipy as sp

# Question 1

Consider buying **April S&P E-mini futures** (or the nearest available contract).

- **Entry condition:** A moving average crossover should signal a buy.
  - Fast moving average: **10 minutes**
  - Slow moving average: **20 minutes**
- **Exit strategy:** Once bought, create a stop-loss exit strategy.
  - **Profit-taking price:** 30 ticks above the buying price.
  - **Stop-loss price:** Calculated using the stop-loss model built in class, with a probability of **0.55** for an up move of one tick.

Submit your ADL diagram.

In [65]:
#pull 1-minute ES futures
es = yf.Ticker("ES=F")

# Get the data
es_data = es.history(period="1d", interval="1m")

# Display the data
print(es_data)

                              Open     High      Low    Close  Volume  \
Datetime                                                                
2026-11-09 00:00:00-05:00  7447.00  7447.00  7444.25  7444.25       0   
2026-11-09 00:01:00-05:00  7444.25  7444.25  7440.50  7441.00     282   
2026-11-09 00:02:00-05:00  7440.00  7441.00  7439.00  7440.00     316   
2026-11-09 00:03:00-05:00  7440.00  7440.50  7439.25  7440.00     190   
2026-11-09 00:04:00-05:00  7440.00  7440.50  7439.25  7439.50     107   
...                            ...      ...      ...      ...     ...   
2026-11-09 13:41:00-05:00  7569.75  7570.50  7568.75  7569.25    1266   
2026-11-09 13:42:00-05:00  7569.50  7571.75  7569.25  7571.00    1328   
2026-11-09 13:43:00-05:00  7571.00  7572.75  7570.50  7572.50    1147   
2026-11-09 13:44:00-05:00  7572.50  7573.75  7571.75  7572.00    1698   
2026-11-09 13:45:00-05:00  7572.00  7573.50  7571.50  7571.75    1707   

                           Dividends  Stock Splits

In [66]:
#calculate moving avgs

fast_t = es_data['Close'].rolling(window=10).mean()
slow_t = es_data['Close'].rolling(window=20).mean()

buy_signals = (fast_t > slow_t) & (fast_t.shift(1) <= slow_t.shift(1))

#we want E to represent the first bar with a buy signal, so we will use the index of the first True value in buy_signals
if buy_signals.any():
    entry_time = buy_signals.idxmax()
    E = es_data.loc[entry_time, 'Close']
else:
    print("No buy signals found.")
    E = None

#stop loss size
r = 0.45/0.55
tick_size = 0.25  # ES futures tick size

alpha = 0.05
L = int(np.ceil(np.log(alpha)/np.log(r)))


#check if the exact probability of being stopped out is about alpha
prob_stop = (r**L - r**(L+30)) / (1 - r**(L+30))
prob_target = 1 - prob_stop
print(f"Probability of being stopped out: {prob_stop:.2f}")
print(f"Probability of reaching the target: {prob_target:.2f}")

#calculat exit prices
TP = E + 30 * tick_size
SL = E - L * tick_size

#simulate the trade
exit_price = None
for t, row in es_data.loc[entry_time:].iloc[1:].iterrows():
    if row['Low'] <= SL:
        print(f"Stopped out at {SL} on {row.name}")
        break
    elif row['High'] >= TP:
        print(f"Target reached at {TP} on {row.name}")
        break

#profit and loss
if exit_price is None:
    exit_price, exit_time = es_data['Close'].iloc[-1], es_data.index[-1]
ticks = (exit_price - E) / tick_size
dollars = ticks * 50  # ES futures contract multiplier

#printed summary of ticks, dollars, P(stop), P(target), E, TP, SL
print(f"Entry Price (E): {E}")
print(f"Target Price (TP): {TP}")
print(f"Stop Loss Price (SL): {SL}")
print(f"Ticks: {ticks:.2f}")
print(f"Dollars: {dollars:.2f}")
print(f"Probability of being stopped out: {prob_stop:.2f}")
print(f"Probability of reaching the target: {prob_target:.2f}")

Probability of being stopped out: 0.05
Probability of reaching the target: 0.95
Stopped out at 7437.5 on 2026-11-09 00:28:00-05:00
Entry Price (E): 7441.25
Target Price (TP): 7448.75
Stop Loss Price (SL): 7437.5
Ticks: 522.00
Dollars: 26100.00
Probability of being stopped out: 0.05
Probability of reaching the target: 0.95


ADD THE ADL DIAGRAM HERE (SOMEONE GO MAKE IT ON ALGO DESIGN LAB PLS)

# Question 4

least cost portfolio of bonds
to purchase today to meet the obligations of the company
over the next 4 years.

In [67]:
liabilities = np.array([12000, 20000, 15000, 20000])
prices = np.array([102, 99, 101, 98, 98])
coupons = np.array([3, 3, 4.5, 3, 4.5])
maturities = np.array([1, 2, 2, 3, 4])

par = 100
reinv_rate = 1.02

cf_matrix = np.zeros((4, 5))

#fill in the cash flow matrix
for i in range(4):
    for j in range(5):
        if i + 1 < maturities[j]:
            cf_matrix[i, j] = coupons[j]
        elif i + 1 == maturities[j]:
            cf_matrix[i, j] = coupons[j] + par
        else:
            cf_matrix[i, j] = 0

print("Cash Flow Matrix: \n",  cf_matrix)

Cash Flow Matrix: 
 [[103.    3.    4.5   3.    4.5]
 [  0.  103.  104.5   3.    4.5]
 [  0.    0.    0.  103.    4.5]
 [  0.    0.    0.    0.  104.5]]


In [68]:
cash_costs = np.array([1, 0, 0, 0])
c = np.concatenate((prices, cash_costs))

cash_block = np.zeros((4, 4))
for i in range(4):
    cash_block[i, i] = reinv_rate
    if i + 1 < 4:
        cash_block[i, i + 1] = -1
    
A_eq = np.hstack([cf_matrix, cash_block])
b_eq = liabilities

A_eq_pd = pd.DataFrame(A_eq)
A_eq_pd.index = ['Year 1', 'Year 2', 'Year 3', 'Year 4']
A_eq_pd.columns = ['x1', 'x2', 'x3', 'x4', 'x5', 's0', 's1', 's2', 's3']

print("A_eq: \n", A_eq_pd)

#test 
test_arr = np.zeros(9)
test_arr[5] = 12000/1.02
test_result = np.dot(A_eq, test_arr)

print("test: \n", test_result)


A_eq: 
            x1     x2     x3     x4     x5    s0    s1    s2    s3
Year 1  103.0    3.0    4.5    3.0    4.5  1.02 -1.00  0.00  0.00
Year 2    0.0  103.0  104.5    3.0    4.5  0.00  1.02 -1.00  0.00
Year 3    0.0    0.0    0.0  103.0    4.5  0.00  0.00  1.02 -1.00
Year 4    0.0    0.0    0.0    0.0  104.5  0.00  0.00  0.00  1.02
test: 
 [12000.     0.     0.     0.]


In [69]:
bounds = [(0, None)] * 9
res = sp.optimize.linprog(c, A_eq=A_eq, b_eq=b_eq, bounds = bounds, method='highs')

first_half = res.x[:5]
second_half = res.x[5:]

res_df = pd.DataFrame({'Variable': ['x1', 'x2', 'x3', 'x4', 'x5', 's0', 's1', 's2', 's3'], 'Value': res.x})
print("Optimization found: \n", res.status)
print("Optimization Message: \n", res.message)
print("Optimization optimal value: \n", res.fun)
print("Optimization Result: \n", res_df)


Optimization found: 
 0
Optimization Message: 
 Optimization terminated successfully. (HiGHS Status 7: Optimal)
Optimization optimal value: 
 60034.12257688657
Optimization Result: 
   Variable        Value
0       x1     0.000000
1       x2     0.000000
2       x3   179.205240
3       x4   137.269476
4       x5   191.387560
5       s0  9726.003897
6       s1     0.000000
7       s2     0.000000
8       s3     0.000000


The minimum cost to fund all four liabilities is $60,034.12: about 179.21 units of Bond 3, 137.27 of Bond 4, 191.39 of Bond 5, and $9,726 held in cash at 2%. Each year's liability is matched exactly, with no surplus carried forward after today. Year 1 is paid from the cash set aside today plus coupons, and Bonds 3, 4 and 5 each mature in time to cover Years 2, 3 and 4. Bond 1 is not used because its one-year return (about 0.98%) is below the 2% reinvestment rate, so cash is the cheaper way to fund Year 1. Bond 2 is not used because Bond 3 also matures in Year 2 and offers a higher yield (about 3.97% vs. 3.53%).

# Question 5

How should you invest to maximize the cash three years
from now?

In [70]:
variable_ordering = ['x1', 'x2', 'x3', 'x4', 's0', 's1', 's2']
size = len(variable_ordering)

#fill in the returns matrix with the results
inv1 = [-1, 0.2, 0, 1.2]
inv2 = [-1, 0.2, 0.9, 0]
inv3 = [0, -1, 0, 1.10]
inv4 = [0, 0, -1, 1.10]
returns_matrix = np.array([inv1, inv2, inv3, inv4])

print("Returns Matrix: \n", returns_matrix)
print("Returns Matrix shape: \n", returns_matrix.shape)



Returns Matrix: 
 [[-1.   0.2  0.   1.2]
 [-1.   0.2  0.9  0. ]
 [ 0.  -1.   0.   1.1]
 [ 0.   0.  -1.   1.1]]
Returns Matrix shape: 
 (4, 4)


In [71]:
# variable ordering: x1, x2, x3, x4, s0, s1, s2

t0 = [1, 1, 0, 0, 1, 0, 0]
t1 = [-0.2, -0.2, 1, 0, -1.05, 1, 0]
t2 = [0, 0.9, 0, -1, 0, 1.05, -1]

A_eq = np.array([t0, t1, t2])

rhs = np.array([1000, 0, 0])

test_plan_a = [0, 0, 0, 0, 1000, 1050, 1102.5]
print("Test Plan A: \n", np.matmul(test_plan_a, A_eq.T))

test_plan_b = [400, 400, 370, 360, 200, 0, 0]
print("Test Plan B: \n", np.matmul(test_plan_b, A_eq.T))

cap = 400
ul_pairs = [(0, cap), (0, cap), (0, cap), (0, cap), (0, None), (0, None), (0, None)]


Test Plan A: 
 [1000.    0.    0.]
Test Plan B: 
 [ 1.0000000e+03 -8.8817842e-15  0.0000000e+00]


In [72]:
coefficients = np.array([1.2, 0, 1.1, 1.1, 0, 0, 1.05])
negated = -coefficients

test_a = np.dot(negated, test_plan_a)
test_b = np.dot(negated, test_plan_b)
print("Test Plan A Objective Value: \n", test_a)
print("Test Plan B Objective Value: \n", test_b)

test = sp.optimize.linprog(negated, A_eq=A_eq, b_eq=rhs, bounds=ul_pairs, method='highs')
print("Optimization found: \n", test.status)
print("Optimization Message: \n", test.message)
print("Optimization Result: \n", test.x)
print("Optimization optimal value: \n", test.fun)

Test Plan A Objective Value: 
 -1157.625
Test Plan B Objective Value: 
 -1283.0
Optimization found: 
 0
Optimization Message: 
 Optimization terminated successfully. (HiGHS Status 7: Optimal)
Optimization Result: 
 [400.  400.    0.  400.  200.  370.  348.5]
Optimization optimal value: 
 -1285.925


The most cash we can have at t = 3 is $1,285.93: invest $400 each in Investments 1, 2 and 4, hold $200 in cash at t = 0, hold $370 at t = 1, and hold $348.50 at t = 2. This is about 11% more than holding all of the cash for three years ($1,157.63). Investment 3 is not used because holding a dollar from t = 1 to t = 3 returns 1.05 × 1.05 = $1.1025 once Investment 4 is at its cap, which is slightly more than Investment 3's $1.10. Investments 1, 2 and 4 are each filled to the $400 limit because each beats holding cash over the same period. Investment 1 is the strongest, returning about $1.42 per dollar, and without the diversification cap the model would put much more into it.

# Question 6

Matrix of foreign exchange rates, determining whether there is an arbitrage opportunity in the currency market

In [73]:
#modeling the currency market as a direct weighted graph

nodes = ['EUR', 'GBP', 'CHF', 'USD', 'AUD', 'CAD', 'HKD', 'INR', 'JPY']
lookup = {node: i for i, node in enumerate(nodes)}
matrix = np.ones((len(nodes), len(nodes)))

#fill in the matrix with actual exchange rates given
exchange_rates = {
    ('EUR', 'GBP'): 0.8725,
    ('EUR', 'CHF'): 1.1379,
    ('EUR', 'USD'): 1.145,
    ('EUR', 'AUD'): 1.5745,
    ('EUR', 'CAD'): 1.5052,
    ('EUR', 'HKD'): 8.9846,
    ('EUR', 'INR'): 81.2662,
    ('EUR', 'JPY'): 124.5984,
    ('GBP', 'CHF'): 1.3041,
    ('GBP', 'USD'): 1.3122,
    ('GBP', 'AUD'): 1.8044,
    ('GBP', 'CAD'): 1.725,
    ('GBP', 'HKD'): 10.2969,
    ('GBP', 'INR'): 93.1355,
    ('GBP', 'JPY'): 142.7965,
    ('CHF', 'USD'): 1.0063,
    ('CHF', 'AUD'): 1.3837,
    ('CHF', 'CAD'): 1.3228,
    ('CHF', 'HKD'): 7.896,
    ('CHF', 'INR'): 71.4194,
    ('CHF', 'JPY'): 109.5011,
    ('USD', 'AUD'): 1.3751,
    ('USD', 'CAD'): 1.3146,
    ('USD', 'HKD'): 7.8469,
    ('USD', 'INR'): 70.975,
    ('USD', 'JPY'): 108.8198,
    ('AUD', 'CAD'): 0.956,
    ('AUD', 'HKD'): 5.7064,
    ('AUD', 'INR'): 51.615,
    ('AUD', 'JPY'): 79.1367,
    ('CAD', 'HKD'): 5.9691,
    ('CAD', 'INR'): 53.9906,
    ('CAD', 'JPY'): 82.7791,
    ('HKD', 'INR'): 9.045,
    ('HKD', 'JPY'): 13.868,
    ('INR', 'JPY'): 1.5332  
}

for (from_currency, to_currency), rate in exchange_rates.items():
    i = lookup[from_currency]
    j = lookup[to_currency]
    matrix[i, j] = rate

#fill in lower triangle of the matrix with reciprocal values
for i in range(len(nodes)):
    for j in range(i):
        matrix[i, j] = 1 / matrix[j, i]


df = pd.DataFrame(matrix, index=nodes, columns=nodes)
print(df)


          EUR       GBP       CHF       USD       AUD       CAD        HKD  \
EUR  1.000000  0.872500  1.137900  1.145000  1.574500  1.505200   8.984600   
GBP  1.146132  1.000000  1.304100  1.312200  1.804400  1.725000  10.296900   
CHF  0.878812  0.766812  1.000000  1.006300  1.383700  1.322800   7.896000   
USD  0.873362  0.762079  0.993739  1.000000  1.375100  1.314600   7.846900   
AUD  0.635122  0.554201  0.722700  0.727220  1.000000  0.956000   5.706400   
CAD  0.664364  0.579710  0.755972  0.760688  1.046025  1.000000   5.969100   
HKD  0.111302  0.097117  0.126646  0.127439  0.175242  0.167529   1.000000   
INR  0.012305  0.010737  0.014002  0.014089  0.019374  0.018522   0.110558   
JPY  0.008026  0.007003  0.009132  0.009190  0.012636  0.012080   0.072108   

           INR       JPY  
EUR  81.266200  124.5984  
GBP  93.135500  142.7965  
CHF  71.419400  109.5011  
USD  70.975000  108.8198  
AUD  51.615000   79.1367  
CAD  53.990600   82.7791  
HKD   9.045000   13.8680  
INR

In [74]:
# arbitrage opportunity if there's a negative cycle. building negative log weight matrix

W = -np.log(matrix)  # weight matrix for Bellman-Ford

W_df = pd.DataFrame(W, index=nodes, columns=nodes)
print(df)


          EUR       GBP       CHF       USD       AUD       CAD        HKD  \
EUR  1.000000  0.872500  1.137900  1.145000  1.574500  1.505200   8.984600   
GBP  1.146132  1.000000  1.304100  1.312200  1.804400  1.725000  10.296900   
CHF  0.878812  0.766812  1.000000  1.006300  1.383700  1.322800   7.896000   
USD  0.873362  0.762079  0.993739  1.000000  1.375100  1.314600   7.846900   
AUD  0.635122  0.554201  0.722700  0.727220  1.000000  0.956000   5.706400   
CAD  0.664364  0.579710  0.755972  0.760688  1.046025  1.000000   5.969100   
HKD  0.111302  0.097117  0.126646  0.127439  0.175242  0.167529   1.000000   
INR  0.012305  0.010737  0.014002  0.014089  0.019374  0.018522   0.110558   
JPY  0.008026  0.007003  0.009132  0.009190  0.012636  0.012080   0.072108   

           INR       JPY  
EUR  81.266200  124.5984  
GBP  93.135500  142.7965  
CHF  71.419400  109.5011  
USD  70.975000  108.8198  
AUD  51.615000   79.1367  
CAD  53.990600   82.7791  
HKD   9.045000   13.8680  
INR

In [75]:
#running Bellman-Ford algorithm to detect negative cycles

import math


distances = np.full(len(nodes), np.inf)
distances[0] = 0
predecessors = np.full(len(nodes), -1)
#need to account for rounding errors in floating point arithmetic, so we will use a small epsilon value
epsilon = 1e-12

for _ in range(len(nodes) - 1):
    for i in range(len(nodes)):
        for j in range(len(nodes)):
            if distances[i] + W[i, j] < distances[j]:
                if distances[i] + W[i, j] < distances[j] - epsilon:
                    distances[j] = distances[i] + W[i, j]
                    predecessors[j] = i

# 9th pass: if any edge can still be relaxed, a negative cycle exists
found = None
for i in range(len(nodes)):
    for j in range(len(nodes)):
        if distances[i] + W[i, j] < distances[j] - epsilon:
            found = j
            break
    if found is not None:
        break

if found is None:
    print("No negative cycle, no arbitrage.")
else:
    print("Negative cycle detected, arbitrage opportunity exists.")

    # phase 1: step back V times to guarantee we're inside the cycle
    current = found
    for _ in range(len(nodes)):
        current = predecessors[current]

    # phase 2: walk around the cycle once, until we return to start
    start = current
    cycle = [start]
    current = predecessors[start]
    while current != start:
        cycle.append(current)
        current = predecessors[current]
    cycle.append(start)
    cycle.reverse()  # predecessors point backwards, so flip to get trade order

    print("Cycle:", " -> ".join(nodes[k] for k in cycle))

#calculating profit from the arbitrage opportunity
profit_factor = 1.0
for i in range(len(cycle) - 1):
    from_currency = nodes[cycle[i]]
    to_currency = nodes[cycle[i + 1]]
    rate = matrix[cycle[i], cycle[i + 1]]
    profit_factor *= rate
    print(f"{from_currency} -> {to_currency} at rate {rate:.4f}")
print(f"Profit: {profit_factor:.7f}")

#weight cross check
cross_check = 1.0
sum = np.sum(W[cycle[i], cycle[i + 1]] for i in range(len(cycle) - 1))
print(f"Cross-check: {math.e**-sum}")
print(f"Return: {(profit_factor - 1) * 100:.7f} percent")



Negative cycle detected, arbitrage opportunity exists.
Cycle: HKD -> JPY -> INR -> HKD
HKD -> JPY at rate 13.8680
JPY -> INR at rate 0.6522
INR -> HKD at rate 0.1106
Profit: 1.0000149
Cross-check: 1.0000148545615832
Return: 0.0014855 percent


/var/folders/v1/nwsdqxmd3sv3nzy2ybgtmykh0000gn/T/ipykernel_63502/2521420208.py:64: DeprecationWarning: Calling np.sum(generator) is deprecated, and in the future will give a different result. Use np.sum(np.fromiter(generator)) or the python sum builtin instead.
  sum = np.sum(W[cycle[i], cycle[i + 1]] for i in range(len(cycle) - 1))


### Interpretation

An arbitrage opportunity does exist since we weighted each edge as −ln(rate), turning an arbitrage cycle (a product of rates greater than 1) into a negative-weight cycle. Using Bellman-Ford:

| Leg | Rate |
|---|---|
| HKD → JPY | 13.868 |
| JPY → INR | 1 / 1.5332 = 0.652231 |
| INR → HKD | 1 / 9.045 = 0.110558 |

Profit factor: 13.868 × 0.652231 × 0.110558 ≈ 1.0000149,
which is a return of about 0.0015% per cycle. Starting with 1,000,000 HKD, one
pass through the cycle ends with about 1,000,014.85 HKD, a profit of about 14.85 HKD.

The direct HKD/JPY quote is slightly higher than the rate implied by going through INR . Converting directly gives slightly more yen than the indirect route, so you buy yen directly and
convert back through INR.